# 面试问题：A2A 与 MCP 有什么区别？Agent Card、Message、Task、Artifact 和生命周期怎样实现？

**一句话回答**：MCP 主要让 Host/Agent 使用工具与资源，A2A 让独立、内部实现可不透明的 Agent 系统互相发现并围绕长任务通信。A2A Server 发布 Agent Card；Client 发送 Message，简单请求可直接返回 Message，长任务返回有状态 Task，并通过 status/artifact events 流式推进到中断或终态。发现能力不等于授权。

本 Notebook 参考 A2A 1.0 规范语义，用标准库手写 Agent Card 校验、能力协商、Task 状态机、stream 顺序、context/task ID、认证边界、幂等重试和多 binding 等价；不是完整网络 SDK。


In [ ]:
from dataclasses import dataclass,field  # 导入本单元所需的依赖。
import hashlib,json,re  # 导入本单元所需的依赖。
from urllib.parse import urlparse  # 导入本单元所需的依赖。

# 协议版本作为会话和 trace 的显式字段，而不是隐藏常量。
PROTOCOL147="1.0.0"; SEED147=14701  # 计算并保存当前步骤的中间状态。
assert PROTOCOL147.count(".")==2  # 用受控断言验证关键不变量。
assert SEED147==14701  # 用受控断言验证关键不变量。
assert len(hashlib.sha256(b"a2a").hexdigest())==64  # 用受控断言验证关键不变量。


## 1. Agent Card 描述接口、能力与认证要求

Card 包含 identity、description、supported interfaces、skills、modalities 与 security schemes，通常从 well-known 地址发现。Card 是不可信远端声明：Client 要 schema 校验、版本/域 allowlist、签名或 registry 信任，且不在卡片中放明文 secret。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AgentCard147:  # 定义承载本节状态与行为的数据结构。
    name:str; endpoint:str; protocol_version:str; interfaces:tuple; skills:tuple; auth_schemes:tuple  # 执行当前语句以推进本节示例。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        # 只接受 HTTPS endpoint，并拒绝把 secret 伪装进 URL query。
        u=urlparse(self.endpoint)  # 计算并保存当前步骤的中间状态。
        if u.scheme!="https" or not u.netloc or u.query: raise ValueError("unsafe_agent_card")  # 按当前条件选择后续控制路径。
card147=AgentCard147("research","https://agents.example/a2a",PROTOCOL147,("JSONRPC","HTTP+JSON"),("research",),("oauth2",))  # 计算并保存当前步骤的中间状态。
assert card147.protocol_version==PROTOCOL147  # 用受控断言验证关键不变量。
assert "research" in card147.skills  # 用受控断言验证关键不变量。
try: AgentCard147("x","http://127.0.0.1?a=secret",PROTOCOL147,(),(),()); raise AssertionError("unsafe card")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="unsafe_agent_card"  # 捕获预期异常并验证失败分支。


## 2. Message 是一次交互，Task 是可跟踪工作单元

Message 由 role 与多个 Parts 组成，可承载 text/file/data。Server 可对简单请求直接回 Agent Message；需要异步、进度、输入或产物的请求创建 server-generated taskId。Client 不能自行猜测不存在的 taskId。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Part147:  # 定义承载本节状态与行为的数据结构。
    kind:str; value:object  # 执行当前语句以推进本节示例。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Message147:  # 定义承载本节状态与行为的数据结构。
    message_id:str; role:str; parts:tuple; context_id:str|None=None; task_id:str|None=None  # 计算并保存当前步骤的中间状态。
def response_mode147(parts):  # 定义本节可复用的核心函数。
    # 长文件/结构化工作进入 Task，短文本可直接 Message 返回。
    return "task" if any(p.kind in {"file","data"} for p in parts) else "message"  # 返回当前分支计算出的结果。
msg147=Message147("m1","user",(Part147("text","你好"),))  # 计算并保存当前步骤的中间状态。
assert response_mode147(msg147.parts)=="message"  # 用受控断言验证关键不变量。
assert response_mode147((Part147("file","ref://1"),))=="task"  # 用受控断言验证关键不变量。
assert msg147.task_id is None  # 用受控断言验证关键不变量。


## 3. Task 状态转换由白名单状态机控制

working 可进入 input-required/auth-required 等中断态，收到合法新消息后回 working；completed/failed/canceled/rejected 是终态，不能继续追加消息。具体枚举以协商版本的规范为准，持久层要用 compare-and-swap 防并发乱序。


In [ ]:
TRANS147={"submitted":{"working","rejected","canceled"},"working":{"input-required","auth-required","completed","failed","canceled"},"input-required":{"working","canceled"},"auth-required":{"working","canceled"}}  # 计算并保存当前步骤的中间状态。
TERMINAL147={"completed","failed","canceled","rejected"}  # 计算并保存当前步骤的中间状态。
@dataclass  # 为下方定义附加声明式配置。
class Task147:  # 定义承载本节状态与行为的数据结构。
    task_id:str; context_id:str; state:str="submitted"; version:int=0  # 计算并保存当前步骤的中间状态。
    def move(self,new):  # 定义本节可复用的核心函数。
        # 终态或非法边不允许继续推进。
        if self.state in TERMINAL147 or new not in TRANS147.get(self.state,set()): raise ValueError("invalid_transition")  # 按当前条件选择后续控制路径。
        self.state=new; self.version+=1  # 计算并保存当前步骤的中间状态。
task147=Task147("t1","c1"); task147.move("working"); task147.move("input-required"); task147.move("working"); task147.move("completed")  # 计算并保存当前步骤的中间状态。
assert task147.state=="completed"  # 用受控断言验证关键不变量。
assert task147.version==4  # 用受控断言验证关键不变量。
try: task147.move("working"); raise AssertionError("terminal reopened")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="invalid_transition"  # 捕获预期异常并验证失败分支。


## 4. Task stream 先给 Task，再给单调更新，终态关闭

如果响应是 Task，stream 首项必须是 Task snapshot；后续 status/artifact update 带 taskId 和递增 sequence。artifact chunk 支持 append，但 final 后不得再写。断线重连用 last seen sequence 去重，而不是重复消费 side effect。


In [ ]:
def validate_stream147(events):  # 定义本节可复用的核心函数。
    # 首事件固定为 task，sequence 严格递增，终态之后禁止事件。
    if not events or events[0]["type"]!="task": return False  # 按当前条件选择后续控制路径。
    last=-1; closed=False  # 计算并保存当前步骤的中间状态。
    for e in events[1:]:  # 遍历输入元素以累积或检查结果。
        if closed or e["seq"]<=last: return False  # 按当前条件选择后续控制路径。
        last=e["seq"]; closed=e.get("state") in TERMINAL147  # 计算并保存当前步骤的中间状态。
    return closed  # 返回当前分支计算出的结果。
events147=[{"type":"task"},{"type":"status","seq":1,"state":"working"},{"type":"artifact","seq":2,"append":True},{"type":"status","seq":3,"state":"completed"}]  # 计算并保存当前步骤的中间状态。
assert validate_stream147(events147)  # 用受控断言验证关键不变量。
assert not validate_stream147(events147+[{"type":"artifact","seq":4}])  # 用受控断言验证关键不变量。
assert not validate_stream147([{"type":"status","seq":1}])  # 用受控断言验证关键不变量。


## 5. contextId 组织会话，taskId 唯一标识一次任务

同一 context 可包含多个 Task 与独立 Messages；继续某个中断 Task 需要同时引用已有 taskId。授权和 TTL 分别检查，不能因为知道 contextId 就读取其中所有 task。服务端生成 ID，并避免在 ID 中编码用户秘密。


In [ ]:
store147={"c1":{"t1":{"owner":"alice"},"t2":{"owner":"alice"}},"c2":{"t3":{"owner":"bob"}}}  # 计算并保存当前步骤的中间状态。
def get_task147(context,task,principal):  # 定义本节可复用的核心函数。
    # context 与 task 必须同时匹配，并再次执行主体授权。
    item=store147.get(context,{}).get(task)  # 计算并保存当前步骤的中间状态。
    if not item or item["owner"]!=principal: raise KeyError("task_not_found")  # 按当前条件选择后续控制路径。
    return item  # 返回当前分支计算出的结果。
assert get_task147("c1","t1","alice")["owner"]=="alice"  # 用受控断言验证关键不变量。
try: get_task147("c1","t3","bob"); raise AssertionError("cross context")  # 尝试执行可能失败的受控操作。
except KeyError as e: assert e.args[0]=="task_not_found"  # 捕获预期异常并验证失败分支。
assert set(store147["c1"])=={"t1","t2"}  # 用受控断言验证关键不变量。


## 6. Agent Card 认证声明与每次授权是两层

Client 按 Card 获取 OAuth/mTLS 等凭据，但 token audience、scope、tenant 和 task ownership 仍由 Server 校验。发现 URL 会产生 SSRF 风险，应限制 scheme/domain/IP、重定向和响应大小。远端 Artifact/Message 一律是不可信数据，不能提升本地指令优先级。


In [ ]:
def safe_discovery147(url,allowed_hosts):  # 定义本节可复用的核心函数。
    # 只允许 HTTPS 且 host 精确命中 allowlist；不做后缀字符串欺骗匹配。
    u=urlparse(url); return u.scheme=="https" and u.hostname in allowed_hosts and not u.username  # 计算并保存当前步骤的中间状态。
assert safe_discovery147("https://agents.example/.well-known/agent-card.json",{"agents.example"})  # 用受控断言验证关键不变量。
assert not safe_discovery147("https://agents.example.evil.test/card",{"agents.example"})  # 用受控断言验证关键不变量。
assert not safe_discovery147("file:///etc/passwd",{"agents.example"})  # 用受控断言验证关键不变量。


## 7. Message retry 需要客户端幂等键，Task ID 仍由服务端生成

网络超时后相同 client message ID 重发应返回同一个 Task/Message 结果；payload 不同却复用 key 必须冲突。Cancel 是 best-effort 状态转换，已完成 Task 返回当前终态，不伪造“物理计算一定瞬间停止”。


In [ ]:
dedup147={}  # 计算并保存当前步骤的中间状态。
def submit147(message_id,payload):  # 定义本节可复用的核心函数。
    # 幂等记录同时保存 payload digest，防 key 被不同请求复用。
    digest=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
    if message_id in dedup147:  # 按当前条件选择后续控制路径。
        if dedup147[message_id][0]!=digest: raise ValueError("idempotency_conflict")  # 按当前条件选择后续控制路径。
        return dedup147[message_id][1]  # 返回当前分支计算出的结果。
    tid="task-"+str(len(dedup147)+1); dedup147[message_id]=(digest,tid); return tid  # 计算并保存当前步骤的中间状态。
assert submit147("m9",{"q":"x"})==submit147("m9",{"q":"x"})  # 用受控断言验证关键不变量。
try: submit147("m9",{"q":"y"}); raise AssertionError("key reuse")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="idempotency_conflict"  # 捕获预期异常并验证失败分支。
assert len(dedup147)==1  # 用受控断言验证关键不变量。


## 8. JSON-RPC、gRPC、HTTP+JSON binding 要保持语义等价

Agent Card 声明支持接口与偏好，Client 选择共同 binding。不同 transport 的 method/status/error 要映射到相同核心 operations；契约测试对同一输入比较 Task 状态、Artifact 与错误语义，而不只比较 HTTP code。


In [ ]:
BINDINGS147={"SendMessage":{"JSONRPC":"SendMessage","GRPC":"SendMessage","HTTP+JSON":"POST /message:send"},"CancelTask":{"JSONRPC":"CancelTask","GRPC":"CancelTask","HTTP+JSON":"POST /tasks/{id}:cancel"}}  # 计算并保存当前步骤的中间状态。
def negotiate147(card,client_supported):  # 定义本节可复用的核心函数。
    # 按 Agent Card 的偏好顺序选择第一个双方支持的 binding。
    return next((x for x in card.interfaces if x in client_supported),None)  # 返回当前分支计算出的结果。
assert negotiate147(card147,{"HTTP+JSON","JSONRPC"})=="JSONRPC"  # 用受控断言验证关键不变量。
assert negotiate147(card147,{"GRPC"}) is None  # 用受控断言验证关键不变量。
assert set(BINDINGS147["SendMessage"])=={"JSONRPC","GRPC","HTTP+JSON"}  # 用受控断言验证关键不变量。


## 面试总结

完整回答是：**A2A 连接独立 Agent、MCP 暴露工具/资源 → 安全获取并校验 Agent Card → 协商 version/binding/auth → Message 可直返或创建 server Task → 白名单状态机支持 input/auth required → Task-first 单调 stream 与 Artifact → contextId/taskId 分工 → 每请求授权/SSRF/不可信数据边界 → message 幂等重试/取消 → 多 binding 契约测试**。

延伸阅读：[A2A 1.0 Specification](https://github.com/a2aproject/A2A/blob/main/docs/specification.md)、[Life of a Task](https://a2aproject.github.io/A2A/latest/topics/life-of-a-task/)、[MCP Architecture](https://modelcontextprotocol.io/docs/learn/architecture)。
